# Assignment #2 
Alia Kasem
All Foundation of Empirical Research 

In [0]:
import requests
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# API URL
url = "https://data.cityofnewyork.us/resource/4b4i-vvec.json?$limit=1000"

resp = requests.get(url)
data = resp.json()

# Convert to pandas then to Spark
pdf = pd.DataFrame(data)
df = spark.createDataFrame(pdf)

df.show(5)


In [0]:
%sql
SHOW SCHEMAS;


In [0]:
%sql 
USE CATALOG workspace;
USE SCHEMA default;



`Air Quality Analysis workflow`

In [0]:
#Python
# Setup Spark
from pyspark.sql import SparkSession
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = SparkSession.builder.getOrCreate()

# Load airquality dataset using Pandas (works with Spark Connect)
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/airquality.csv"
pdf = pd.read_csv(url)

# Drop index column
pdf = pdf.drop(columns=["Unnamed: 0"], errors='ignore')

# Convert to Spark DataFrame
df = spark.createDataFrame(pdf)

# Save as Spark table
df.write.mode("overwrite").saveAsTable("airquality")

display(df)


`SQL Queries` 

In [0]:
%sql
--show frist 10 rows 
SELECT * 
FROM airquality
LIMIT 10;


In [0]:
%sql 
-- count of records
SELECT Month, COUNT(*) AS num_records
FROM airquality
GROUP BY Month
ORDER BY Month ASC;


In [0]:
%sql 
-- Average AQI by month
SELECT Month, AVG(Ozone) AS Ozone
FROM airquality
GROUP BY Month
ORDER BY Month ASC

In [0]:
%sql 
-- Max temperature by month
SELECT Month, MAX(Temp) AS Max_Temp
FROM airquality
GROUP BY Month
ORDER BY Month ASC

## `Pandas Descriptive Statistics` 

In [0]:
# convert to pandas
pdf = df.toPandas()

# descriptive statistics
pdf.describe()
# Python
# Setup Spark
from pyspark.sql import SparkSession
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

spark = SparkSession.builder.getOrCreate()

# Drop index column
pdf = pdf.drop(columns=["Unnamed: 0"], errors='ignore')
# Convert to Spark DataFrame
df = spark.createDataFrame(pdf)
# Save as Spark table
df.write.mode("overwrite").saveAsTable("airquality")
display(df)
# convert to pandas
pdf = df.toPandas()
# descriptive statistics
pdf.describe()
# Python
# Setup Spark
from pyspark.sql import SparkSession
import pandas as pd
import matplotlib.pyplot as plt 

In [0]:
# Descriptive stats
desc_stats = pdf.describe()
print(desc_stats)


In [0]:
# Get unique months in the dataset
df.select("Month").distinct().orderBy("Month").show()


In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Map month numbers to names
month_map = {5:'May', 6:'June', 7:'July', 8:'August', 9:'September'}
pdf['MonthName'] = pdf['Month'].map(month_map)

# Average Ozone by Month 5-9 only 
avg_ozone = pdf.groupby('MonthName')['Ozone'].mean().reindex(month_map.values()).reset_index()

plt.figure(figsize=(8,5))
sns.barplot(x='MonthName', y='Ozone', data=avg_ozone, palette='cool')
plt.title('Average Ozone Levels by Month')
plt.xlabel('Month')
plt.ylabel('Ozone')
plt.show()


`Figure 1. Avg Ozone level by month, indicating the month.`

In [0]:
plt.figure(figsize=(8,5))
sns.scatterplot(x='Temp', y='Ozone', hue='MonthName', data=pdf, palette='viridis', s=70)
plt.title('Ozone vs Temperature')
plt.xlabel('Temperature (F)')
plt.ylabel('Ozone')
plt.legend(title='Month')
plt.show()

`Figue 2. The level of Ozone in comparison to the temperature, indicating Augest to have the highest temperature impacting the Ozone.`

In [0]:
avg_wind = pdf.groupby('MonthName')['Wind'].mean().reindex(month_map.values()).reset_index()

plt.figure(figsize=(8,5))
sns.lineplot(x='MonthName', y='Wind', data=avg_wind, marker='o', color='blue')
plt.title('Average Wind by Month')
plt.xlabel('Month')
plt.ylabel('Wind (mph)')
plt.show()

`Figure 3. The average wind per month, indicating Augest to have lowest wind mph. `

In [0]:
sns.pairplot(pdf[['Ozone','Solar.R','Wind','Temp']], diag_kind='kde', plot_kws={'alpha':0.6})
plt.suptitle('Pairplot of Ozone, Solar Radiation, Wind, and Temperature', y=1.02)
plt.show()

In [0]:
# Solar radiation vs Temperature (scatter + regression line)
plt.figure(figsize=(8,5))
sns.regplot(x='Solar.R', y='Temp', data=pdf, scatter_kws={'s':50}, line_kws={'color':'red'})
plt.title('Solar Radiation vs Temperature')
plt.xlabel('Solar Radiation')
plt.ylabel('Temperature (F)')
plt.show()



`The scatter plot with a regression line shows a positive relationship, indicating that higher levels of solar radiation are generally associated with higher temperatures; however, the plot indicates other factors that influenced the temperature beyond the solar radiation, such as the wind, humidity, and time of day.`